# EDGE AI CHALLENGE 2026 — YOLO + TextCNN (Kaggle)

Notebook này xây dựng hai pipeline:
1. **Object Detection:** huấn luyện YOLO từ ảnh + nhãn YOLO.
2. **Intent Classification:** huấn luyện TextCNN trên `intent_train.csv`.
3. Chạy dự đoán test và tạo `/kaggle/working/submission.csv`.

> **Trước khi chạy:** mở Notebook trên Kaggle, bật Internet nếu được phép để tải package/trọng số YOLO. Trong phần **Add Input**, thêm dataset cuộc thi. Sửa `INPUT_DIR` ở Cell cấu hình cho đúng đường dẫn hiển thị trong `/kaggle/input`.
>
> Notebook mặc định dùng Ultralytics YOLO để làm baseline Warm-up. Đây chưa phải cam kết đáp ứng giới hạn 200.000 tham số của vòng loại/chung kết; cần kiểm tra và tối ưu riêng trước khi triển khai ESP32-S3.


## 1. Cài đặt thư viện

In [ ]:
# Nếu Ultralytics đã có sẵn trong môi trường thì có thể bỏ qua cell này.
%pip install -q ultralytics


## 2. Import và cấu hình

In [ ]:
import os
import re
import json
import csv
import random
import shutil
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)


## 3. Tìm và khai báo dataset
Chạy cell kế tiếp để xem thư mục được Kaggle gắn vào. Sau đó sửa `INPUT_DIR` nếu cần.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
for p in INPUT_ROOT.iterdir():
    print(p)

# SỬA đường dẫn này theo dataset thực tế của cuộc thi:
INPUT_DIR = Path("/kaggle/input/edge-ai-challenge-2026")

WORK_DIR = Path("/kaggle/working")
YOLO_DATA_DIR = WORK_DIR / "yolo_dataset"
RUNS_DIR = WORK_DIR / "yolo_runs"
OUT_DIR = WORK_DIR / "outputs"
CKPT_DIR = WORK_DIR / "checkpoints"

for d in [OUT_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_IMAGE_DIR = INPUT_DIR / "train" / "images"
TRAIN_LABEL_DIR = INPUT_DIR / "train" / "labels"
TEST_IMAGE_DIR = INPUT_DIR / "test" / "images"
CLASSES_FILE = INPUT_DIR / "classes.txt"
INTENT_TRAIN_FILE = INPUT_DIR / "intent_train.csv"
INTENT_TEST_FILE = INPUT_DIR / "intent_test.csv"
SAMPLE_SUBMISSION_FILE = INPUT_DIR / "sample_submission.csv"

required_paths = [
    TRAIN_IMAGE_DIR, TRAIN_LABEL_DIR, TEST_IMAGE_DIR,
    CLASSES_FILE, INTENT_TRAIN_FILE, INTENT_TEST_FILE
]
for p in required_paths:
    print(("OK      " if p.exists() else "MISSING "), p)

if not all(p.exists() for p in required_paths):
    print("\nHãy sửa INPUT_DIR cho đúng. Không chạy tiếp cho đến khi mọi đường dẫn hiện OK.")


## 4. Đọc danh sách lớp và kiểm tra dữ liệu

In [ ]:
with open(CLASSES_FILE, "r", encoding="utf-8") as f:
    classes = [line.strip() for line in f if line.strip()]

NUM_CLASSES = len(classes)
print("Số lớp:", NUM_CLASSES)
for i, name in enumerate(classes):
    print(i, name)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
train_images = sorted(
    p for p in TRAIN_IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
)
test_images = sorted(
    p for p in TEST_IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
)

print(f"Train images: {len(train_images)}")
print(f"Test images : {len(test_images)}")
print(f"Train labels: {len(list(TRAIN_LABEL_DIR.glob('*.txt')))}")

missing_labels = [
    p.name for p in train_images
    if not (TRAIN_LABEL_DIR / f"{p.stem}.txt").exists()
]
print("Ảnh thiếu file label:", len(missing_labels), missing_labels[:10])


## 5. Kiểm tra nhãn YOLO và trực quan hóa
Định dạng nhãn phải là `class_id cx cy w h`, tọa độ chuẩn hóa 0–1. File `.txt` rỗng là ảnh nền; file thiếu cần được điều tra.

In [ ]:
def read_yolo_label(label_path):
    rows = []
    if not label_path.exists():
        return rows
    text = label_path.read_text(encoding="utf-8").strip()
    if not text:
        return rows
    for line_no, line in enumerate(text.splitlines(), start=1):
        vals = line.split()
        if len(vals) != 5:
            raise ValueError(f"{label_path}, dòng {line_no}: cần 5 giá trị")
        cls, cx, cy, w, h = map(float, vals)
        if not (0 <= int(cls) < NUM_CLASSES):
            raise ValueError(f"class_id ngoài phạm vi trong {label_path}: {cls}")
        if not all(0 <= v <= 1 for v in [cx, cy, w, h]):
            raise ValueError(f"Tọa độ YOLO ngoài [0,1] trong {label_path}")
        rows.append((int(cls), cx, cy, w, h))
    return rows

counter = Counter()
empty_count = 0
for p in train_images:
    lp = TRAIN_LABEL_DIR / f"{p.stem}.txt"
    if not lp.exists():
        continue
    labels = read_yolo_label(lp)
    if not labels:
        empty_count += 1
    counter.update(x[0] for x in labels)

print("Ảnh có label rỗng:", empty_count)
print("Số box theo lớp:")
for i, name in enumerate(classes):
    print(f"{i:2d} {name:35s} {counter[i]}")

def show_yolo_image(image_path):
    image_path = Path(image_path)
    im = Image.open(image_path).convert("RGB")
    W, H = im.size
    draw = ImageDraw.Draw(im)
    labels = read_yolo_label(TRAIN_LABEL_DIR / f"{image_path.stem}.txt")
    for cls, cx, cy, w, h in labels:
        x1 = (cx - w/2) * W
        y1 = (cy - h/2) * H
        x2 = (cx + w/2) * W
        y2 = (cy + h/2) * H
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, max(0, y1-14)), f"{cls}: {classes[cls]}", fill="yellow")
    plt.figure(figsize=(8, 6))
    plt.imshow(im)
    plt.axis("off")
    plt.show()

for p in train_images[:5]:
    show_yolo_image(p)


## 6. Chia train/validation và chuyển sang cấu trúc YOLO

Ultralytics cần file YAML và thư mục ảnh/nhãn tương ứng. Cell này tạo bản sao trong `/kaggle/working/yolo_dataset`; dữ liệu gốc ở `/kaggle/input` không bị thay đổi.

Để tránh lỗi do thiếu file nhãn, cell sẽ **dừng** nếu có ảnh không có file `.txt`. Nếu BTC xác nhận ảnh đó là ảnh nền, hãy tạo file nhãn rỗng tương ứng trong bản sao.


In [ ]:
if missing_labels:
    raise FileNotFoundError(
        f"Có {len(missing_labels)} ảnh thiếu file nhãn. Kiểm tra dữ liệu trước khi train. "
        "Không tự động coi file thiếu là ảnh nền."
    )

# Tách ảnh theo seed cố định. Với dữ liệu nhỏ, hãy kiểm tra phân bố lớp ở validation.
train_paths, val_paths = train_test_split(
    train_images, test_size=0.2, random_state=SEED, shuffle=True
)

for split in ["train", "val"]:
    (YOLO_DATA_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATA_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def copy_yolo_split(paths, split):
    for img_path in paths:
        label_path = TRAIN_LABEL_DIR / f"{img_path.stem}.txt"
        shutil.copy2(img_path, YOLO_DATA_DIR / "images" / split / img_path.name)
        shutil.copy2(label_path, YOLO_DATA_DIR / "labels" / split / label_path.name)

copy_yolo_split(train_paths, "train")
copy_yolo_split(val_paths, "val")

# YAML dùng đường dẫn tuyệt đối
yaml_path = YOLO_DATA_DIR / "data.yaml"
names_yaml = "\n".join([f"  {i}: {json.dumps(name, ensure_ascii=False)}" for i, name in enumerate(classes)])
yaml_text = (
    f"path: {YOLO_DATA_DIR}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"names:\n{names_yaml}\n"
)
yaml_path.write_text(yaml_text, encoding="utf-8")
print(yaml_text)
print("Train:", len(train_paths), "Validation:", len(val_paths))


## 7. Huấn luyện YOLO
Baseline dùng YOLO nano pretrained. Kaggle cần Internet hoặc bạn phải thêm trọng số vào Input. `imgsz=160` thử nghiệm giới hạn đầu vào; Warm-up có thể thử kích thước khác.

In [ ]:
# Nếu không thể tải trọng số, thêm file .pt làm Kaggle Input rồi sửa MODEL_WEIGHTS.
MODEL_WEIGHTS = "yolo11n.pt"
YOLO_EPOCHS = 100
YOLO_IMGSZ = 160
YOLO_BATCH = 16

detector = YOLO(MODEL_WEIGHTS)
train_result = detector.train(
    data=str(yaml_path),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMGSZ,
    batch=YOLO_BATCH,
    patience=20,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=2,
    seed=SEED,
    project=str(RUNS_DIR),
    name="warmup_yolo",
    exist_ok=True,
    plots=True,
    verbose=True
)

BEST_PT = RUNS_DIR / "warmup_yolo" / "weights" / "best.pt"
LAST_PT = RUNS_DIR / "warmup_yolo" / "weights" / "last.pt"
print("Best checkpoint exists:", BEST_PT.exists(), BEST_PT)


## 8. Đánh giá YOLO trên validation

In [ ]:
best_detector = YOLO(str(BEST_PT))
val_metrics = best_detector.val(
    data=str(yaml_path),
    imgsz=YOLO_IMGSZ,
    device=0 if torch.cuda.is_available() else "cpu",
    split="val"
)
print(val_metrics.results_dict)


## 9. Dự đoán tập test bằng YOLO

In [ ]:
# Ngưỡng thấp để giữ lại nhiều ứng viên; điều chỉnh dựa trên validation.
DET_CONF = 0.001
DET_IOU = 0.7

test_results = best_detector.predict(
    source=[str(p) for p in test_images],
    imgsz=YOLO_IMGSZ,
    conf=DET_CONF,
    iou=DET_IOU,
    save=False,
    verbose=False,
    device=0 if torch.cuda.is_available() else "cpu"
)

detection_predictions = {}
original_sizes = {}

for result in test_results:
    image_path = Path(result.path)
    sample_id = image_path.stem
    with Image.open(image_path) as im:
        orig_w, orig_h = im.size
    original_sizes[sample_id] = (orig_w, orig_h)

    preds = []
    if result.boxes is not None:
        for box in result.boxes:
            cls_id = int(box.cls.item())
            score = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            # Làm tròn và giới hạn trong ảnh gốc
            x1 = max(0, min(orig_w, int(round(x1))))
            y1 = max(0, min(orig_h, int(round(y1))))
            x2 = max(0, min(orig_w, int(round(x2))))
            y2 = max(0, min(orig_h, int(round(y2))))

            if x2 <= x1 or y2 <= y1:
                continue

            preds.append({
                "class_id": cls_id,
                "score": round(max(0.0, min(1.0, score)), 6),
                "bbox": [x1, y1, x2, y2]
            })

    detection_predictions[sample_id] = preds

print("Số ảnh đã dự đoán:", len(detection_predictions))
print("Ví dụ:", next(iter(detection_predictions.items()), None))


## 10. Chuẩn bị dữ liệu Intent

Mô hình bắt buộc là TextCNN, không dùng luật từ khóa thay thế. Do chỉ có ít dữ liệu, validation sẽ dao động; không nên diễn giải điểm validation như bảo đảm cho test.

In [ ]:
intent_train = pd.read_csv(INTENT_TRAIN_FILE)
intent_test = pd.read_csv(INTENT_TEST_FILE)

required_train_cols = {"question_id", "question", "intent"}
required_test_cols = {"question_id", "question"}
assert required_train_cols.issubset(intent_train.columns), intent_train.columns
assert required_test_cols.issubset(intent_test.columns), intent_test.columns

INTENT_LABELS = ["COUNT", "EXIST", "LIST", "LOCATION", "OUT_OF_SCOPE"]
assert intent_train["question"].notna().all()
assert intent_train["intent"].notna().all()
unknown_labels = set(intent_train["intent"].unique()) - set(INTENT_LABELS)
assert not unknown_labels, f"Nhãn không hợp lệ: {unknown_labels}"

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9_\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

intent_train["question_clean"] = intent_train["question"].map(normalize_text)
intent_test["question_clean"] = intent_test["question"].map(normalize_text)

print("Train rows:", len(intent_train))
print("Test rows:", len(intent_test))
print(intent_train["intent"].value_counts())
display(intent_train.head())


## 11. Chia train/validation và tạo vocabulary

In [ ]:
# Stratify có thể lỗi nếu một nhãn quá ít mẫu; khi đó cần điều chỉnh cách chia.
try:
    intent_tr_df, intent_val_df = train_test_split(
        intent_train,
        test_size=0.2,
        random_state=SEED,
        stratify=intent_train["intent"]
    )
except ValueError as e:
    print("Không thể stratify:", e)
    print("Dùng chia ngẫu nhiên thay thế; hãy lưu ý validation có thể thiếu lớp.")
    intent_tr_df, intent_val_df = train_test_split(
        intent_train, test_size=0.2, random_state=SEED, shuffle=True
    )

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
token_counts = Counter()
for text in intent_tr_df["question_clean"]:
    token_counts.update(text.split())

vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for token in token_counts:
    if token not in vocab:
        vocab[token] = len(vocab)

label_to_id = {label: i for i, label in enumerate(INTENT_LABELS)}
id_to_label = {i: label for label, i in label_to_id.items()}

MAX_LEN = max(
    8,
    min(32, max(len(x.split()) for x in intent_train["question_clean"]) + 2)
)

print("Vocabulary:", len(vocab), "MAX_LEN:", MAX_LEN)
print("Train/val:", len(intent_tr_df), len(intent_val_df))

def encode_text(text):
    ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in text.split()]
    ids = ids[:MAX_LEN]
    ids += [vocab[PAD_TOKEN]] * (MAX_LEN - len(ids))
    return ids

class IntentDataset(Dataset):
    def __init__(self, df, labeled=True):
        self.texts = df["question_clean"].tolist()
        self.labeled = labeled
        self.labels = (
            [label_to_id[x] for x in df["intent"].tolist()]
            if labeled else None
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode_text(self.texts[idx]), dtype=torch.long)
        if self.labeled:
            return x, torch.tensor(self.labels[idx], dtype=torch.long)
        return x

BATCH_SIZE = 8
train_loader = DataLoader(
    IntentDataset(intent_tr_df, labeled=True),
    batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    IntentDataset(intent_val_df, labeled=True),
    batch_size=BATCH_SIZE, shuffle=False
)


## 12. Định nghĩa TextCNN

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, num_filters=24,
                 kernel_sizes=(2, 3, 4), num_classes=5, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=0
        )
        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, input_ids):
        # [batch, seq] -> [batch, seq, emb] -> [batch, emb, seq]
        x = self.embedding(input_ids).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            z = torch.relu(conv(x))
            z = torch.max(z, dim=2).values
            pooled.append(z)
        x = torch.cat(pooled, dim=1)
        x = self.dropout(x)
        return self.fc(x)

textcnn = TextCNN(
    vocab_size=len(vocab),
    embedding_dim=32,
    num_filters=24,
    kernel_sizes=(2, 3, 4),
    num_classes=len(INTENT_LABELS),
    dropout=0.3
).to(DEVICE)

print(textcnn)
print("Parameters:", sum(p.numel() for p in textcnn.parameters()))


## 13. Huấn luyện TextCNN và lưu checkpoint tốt nhất

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(textcnn.parameters(), lr=1e-3)

EPOCHS = 100
PATIENCE = 12
best_val_acc = -1.0
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    textcnn.train()
    train_losses = []

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = textcnn(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    textcnn.eval()
    val_true, val_pred = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            logits = textcnn(xb)
            preds = logits.argmax(dim=1).cpu().tolist()
            val_pred.extend(preds)
            val_true.extend(yb.tolist())

    val_acc = accuracy_score(val_true, val_pred) if val_true else 0.0
    train_loss = float(np.mean(train_losses)) if train_losses else 0.0
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_acc
    })

    print(f"Epoch {epoch:03d} | loss={train_loss:.4f} | val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in textcnn.state_dict().items()
        }
        torch.save({
            "model_state_dict": best_state,
            "vocab": vocab,
            "label_to_id": label_to_id,
            "max_len": MAX_LEN,
            "classes": INTENT_LABELS
        }, CKPT_DIR / "best_textcnn.pth")
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

if best_state is not None:
    textcnn.load_state_dict(best_state)

pd.DataFrame(history).to_csv(OUT_DIR / "textcnn_training_history.csv", index=False)
print("Best validation accuracy:", best_val_acc)


## 14. Đánh giá TextCNN trên validation

In [ ]:
textcnn.eval()
val_true, val_pred = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        logits = textcnn(xb.to(DEVICE))
        val_pred.extend(logits.argmax(dim=1).cpu().tolist())
        val_true.extend(yb.tolist())

print("Validation accuracy:", accuracy_score(val_true, val_pred))
print(classification_report(
    val_true, val_pred,
    labels=list(range(len(INTENT_LABELS))),
    target_names=INTENT_LABELS,
    zero_division=0
))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(
    val_true, val_pred,
    labels=list(range(len(INTENT_LABELS)))
))


## 15. Dự đoán intent_test.csv

In [ ]:
test_intent_ds = IntentDataset(intent_test.assign(
    question_clean=intent_test["question_clean"]
), labeled=False)
test_intent_loader = DataLoader(
    test_intent_ds, batch_size=32, shuffle=False
)

intent_predictions = []
textcnn.eval()
with torch.no_grad():
    for xb in test_intent_loader:
        logits = textcnn(xb.to(DEVICE))
        pred_ids = logits.argmax(dim=1).cpu().tolist()
        intent_predictions.extend(id_to_label[i] for i in pred_ids)

assert len(intent_predictions) == len(intent_test)
intent_prediction_dict = dict(zip(
    intent_test["question_id"].astype(str),
    intent_predictions
))

intent_pred_df = intent_test[["question_id", "question"]].copy()
intent_pred_df["prediction"] = intent_predictions
intent_pred_df.to_csv(OUT_DIR / "intent_predictions.csv", index=False)
display(intent_pred_df.head())


## 16. Tạo submission.csv

In [ ]:
submission_rows = []

# DETECTION: mỗi ảnh test phải xuất hiện đúng một lần
for image_path in test_images:
    sample_id = image_path.stem
    preds = detection_predictions.get(sample_id, [])
    submission_rows.append({
        "task_type": "DETECTION",
        "sample_id": sample_id,
        "prediction": json.dumps(
            preds, ensure_ascii=False, separators=(",", ":")
        )
    })

# INTENT: mỗi question_id test phải xuất hiện đúng một lần
for _, row in intent_test.iterrows():
    qid = str(row["question_id"])
    submission_rows.append({
        "task_type": "INTENT",
        "sample_id": qid,
        "prediction": intent_prediction_dict[qid]
    })

submission = pd.DataFrame(
    submission_rows,
    columns=["task_type", "sample_id", "prediction"]
)
submission_path = WORK_DIR / "submission.csv"
submission.to_csv(
    submission_path,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL
)

print("Saved:", submission_path)
print("Rows:", len(submission))
display(submission.head(10))


## 17. Kiểm tra định dạng submission

In [ ]:
check = pd.read_csv(submission_path, dtype=str)

assert check.columns.tolist() == ["task_type", "sample_id", "prediction"]
assert len(check) == len(test_images) + len(intent_test)
assert not check["sample_id"].duplicated().any()
assert set(check["task_type"].unique()) == {"DETECTION", "INTENT"}

det_rows = check[check["task_type"] == "DETECTION"]
int_rows = check[check["task_type"] == "INTENT"]

assert set(det_rows["sample_id"]) == {p.stem for p in test_images}
assert set(int_rows["sample_id"]) == set(intent_test["question_id"].astype(str))
assert set(int_rows["prediction"]).issubset(set(INTENT_LABELS))

for _, row in det_rows.iterrows():
    preds = json.loads(row["prediction"])
    assert isinstance(preds, list) and len(preds) <= 100
    image_size = original_sizes[row["sample_id"]]
    W, H = image_size
    for obj in preds:
        assert set(obj.keys()) == {"class_id", "score", "bbox"}
        cls_id = int(obj["class_id"])
        score = float(obj["score"])
        bbox = obj["bbox"]
        assert 0 <= cls_id < NUM_CLASSES
        assert 0 <= score <= 1
        assert isinstance(bbox, list) and len(bbox) == 4
        assert all(isinstance(v, int) for v in bbox)
        x1, y1, x2, y2 = bbox
        assert 0 <= x1 < x2 <= W
        assert 0 <= y1 < y2 <= H

print("PASS: kiểm tra cơ bản thành công.")
print("DETECTION:", len(det_rows), "| INTENT:", len(int_rows))
print("Submission:", submission_path)


## 18. Tải file và lưu ý sử dụng

Khi Notebook chạy xong, file nộp nằm ở **`/kaggle/working/submission.csv`**. Trong Kaggle, mở mục Output/Files của Notebook để tải file hoặc dùng trực tiếp nếu cuộc thi cho phép nộp từ output.

### Lưu ý quan trọng
- Kiểm tra `INPUT_DIR` trước khi chạy.
- Nếu Kaggle không có Internet, thêm trọng số YOLO `.pt` vào Input và cập nhật `MODEL_WEIGHTS`.
- Warm-up dùng 10 lớp theo `classes.txt`; vòng loại phải đổi theo bộ 7 lớp mới.
- Kết quả TextCNN với 63 câu train có thể dao động mạnh theo cách chia validation.
- YOLO nano baseline có thể vượt giới hạn 200.000 tham số; muốn triển khai ESP32-S3 cần thiết kế/đổi sang detector nhỏ hơn, lượng tử hóa INT8 và kiểm tra operator/RAM.
- Hãy đối chiếu `submission.csv` với `sample_submission.csv` và `submission_format.txt` của BTC trước khi nộp.
